In [ ]:
import pandas as pd
import numpy as np
import re
from scipy.stats import wilcoxon, binomtest

pd.set_option("display.max_columns", 50)

In [ ]:
df_raw = pd.read_csv('../human_eval_data.csv')
print(df_raw.shape)
df_raw.head()

In [ ]:
# clean data
df = df_raw.copy()

df["Progress"] = df["Progress"].astype(str)
df = df[df["Progress"] == "100"]

drop_cols = ['StartDate', 'EndDate', 'Status', 'IPAddress', 'Progress',
             'Duration (in seconds)', 'Finished', 'RecordedDate', 'ResponseId',
             'RecipientLastName', 'RecipientFirstName', 'RecipientEmail',
             'ExternalReference', 'LocationLatitude', 'LocationLongitude',
             'DistributionChannel', 'UserLanguage', 'Q5', 'Q6', 'Q7']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df = df.reset_index(drop=True)
df.insert(0, "participant_id", df.index)

print(len(df))
df.head()

In [ ]:
N_PAIRS = 5
REFINED_LETTER_BY_PAIR = list("ABBAA")   

RATING_ORDER = ["MC_A", "AA_A", "MC_B", "AA_B"]

In [ ]:
rating_cols = [c for c in df.columns if re.fullmatch(r"Q\d+_1", c)]
choice_cols = [c for c in df.columns if re.fullmatch(r"Q\d+", c)
               and df[c].dropna().isin(["Image A", "Image B"]).any()]

pair_id_cols = [f"pair_{i}_id" for i in range(1, N_PAIRS + 1)]
pair_metaphor_cols = [f"pair_{i}_metaphor" for i in range(1, N_PAIRS + 1)]

print("Rating columns (in order):", rating_cols)
print("Choice columns (in order):", choice_cols)

In [ ]:
# reformat
records = []

for pair_idx in range(N_PAIRS):
    pair_num = pair_idx + 1
    refined_letter = REFINED_LETTER_BY_PAIR[pair_idx]
    zero_letter = "B" if refined_letter == "A" else "A"

    block_rating_cols = rating_cols[pair_idx * 4: (pair_idx + 1) * 4]
    choice_col = choice_cols[pair_idx]
    id_col = pair_id_cols[pair_idx]
    metaphor_col = pair_metaphor_cols[pair_idx]

    col_by_label = dict(zip(RATING_ORDER, block_rating_cols))

    for _, row in df.iterrows():
        chosen_image = row[choice_col]  # "Image A" or "Image B"
        chosen_condition = "refined" if chosen_image == f"Image {refined_letter}" else "zero_shot"

        for axis in ["MC", "AA"]:
            for letter, condition in [(refined_letter, "refined"), (zero_letter, "zero_shot")]:
                score_col = col_by_label[f"{axis}_{letter}"]
                records.append({
                    "participant_id": row["participant_id"],
                    "pair_num": pair_num,
                    "metaphor_id": row.get(id_col, np.nan),
                    "metaphor_text": row.get(metaphor_col, np.nan),
                    "axis": axis,
                    "condition": condition,
                    "score": row[score_col],
                    "preferred_condition": chosen_condition,
                })

long_df = pd.DataFrame(records)
long_df["score"] = pd.to_numeric(long_df["score"], errors="coerce")

print(long_df.shape)
long_df.head(10)

In [ ]:
# descriptive statistics
desc = (
    long_df.groupby(["axis", "condition"])["score"]
    .agg(n="count", mean="mean", median="median", sd="std", min="min", max="max")
    .round(3)
)
desc

In [ ]:
# wilcoxon signed rank
wide = long_df.pivot_table(
    index=["participant_id", "pair_num", "axis"],
    columns="condition",
    values="score"
).reset_index()

results = {}
for axis in ["MC", "AA"]:
    sub = wide[wide["axis"] == axis].dropna(subset=["zero_shot", "refined"])
    stat, p = wilcoxon(sub["refined"], sub["zero_shot"])
    results[axis] = {
        "n_pairs": len(sub),
        "W": stat,
        "p": p,
        "median_diff_refined_minus_zero": (sub["refined"] - sub["zero_shot"]).median(),
    }

wilcoxon_df = pd.DataFrame(results).T
# Bonferroni correction across the 2 axes
wilcoxon_df["p_bonferroni"] = (wilcoxon_df["p"] * len(wilcoxon_df)).clip(upper=1.0)
wilcoxon_df

In [ ]:
# forced choice results
pref_df = long_df.drop_duplicates(subset=["participant_id", "pair_num"])[
    ["participant_id", "pair_num", "preferred_condition"]
]

pref_counts = pref_df["preferred_condition"].value_counts(normalize=True).round(3)
print(pref_counts)

n_refined = (pref_df["preferred_condition"] == "refined").sum()
n_total = len(pref_df)
result = binomtest(n_refined, n_total, p=0.5)
print(f"\nRefined preferred in {n_refined}/{n_total} ({n_refined/n_total:.1%}) "
      f"— binomial test p = {result.pvalue:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ============================================================
# 1. Classify MC / AA comparisons
# ============================================================

wide = long_df.pivot_table(
    index=["participant_id", "pair_num", "axis"],
    columns="condition",
    values="score"
).reset_index()


def classify_rating(row):
    refined = row["refined"]
    zero = row["zero_shot"]

    if pd.isna(refined) or pd.isna(zero):
        return np.nan

    if refined > zero:
        return "Refined better"
    elif zero > refined:
        return "Zero better"
    else:
        # Equal scores
        if refined >= 7:
            return "Equally good"
        else:
            return "Equally bad"


wide["outcome"] = wide.apply(classify_rating, axis=1)


# ============================================================
# 2. Calculate MC / AA proportions
# ============================================================

rating_summary = (
    wide.groupby(["axis", "outcome"])
    .size()
    .reset_index(name="count")
)

rating_summary["proportion"] = (
    rating_summary.groupby("axis")["count"]
    .transform(lambda x: x / x.sum())
)

rating_summary = rating_summary.drop(columns="count")


# ============================================================
# 3. Forced-choice comparison
# ============================================================

forced = (
    long_df[
        ["participant_id", "pair_num", "preferred_condition"]
    ]
    .drop_duplicates()
)

forced["axis"] = "Forced choice"

forced["outcome"] = forced["preferred_condition"].map({
    "refined": "Refined better",
    "zero_shot": "Zero better"
})


forced_summary = (
    forced.groupby(["axis", "outcome"])
    .size()
    .reset_index(name="count")
)

forced_summary["proportion"] = (
    forced_summary.groupby("axis")["count"]
    .transform(lambda x: x / x.sum())
)

forced_summary = forced_summary.drop(columns="count")


# ============================================================
# 4. Combine data
# ============================================================

plot_df = pd.concat(
    [
        rating_summary,
        forced_summary
    ],
    ignore_index=True
)


# Plot order
axes_order = [
    "AA",
    "MC",
    "Forced choice"
]

# Display labels
axis_labels = [
    "Analogy\nAppropriateness",
    "Metaphor\nConsistency",
    "Forced\nchoice"
]


categories = [
    "Refined better",
    "Equally good",
    "Equally bad",
    "Zero better"
]


# ============================================================
# 5. Plot 100% stacked bar chart
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))


# Custom colours
colors = {
    "Refined better": "#4C72B0",   # blue
    "Equally good": "#55A868",     # green
    "Equally bad": "#BDBDBD",      # grey
    "Zero better": "#C44E52"       # red
}


bottom = np.zeros(len(axes_order))

for category in categories:

    values = []

    for axis in axes_order:

        subset = plot_df[
            (plot_df["axis"] == axis) &
            (plot_df["outcome"] == category)
        ]

        if len(subset) > 0:
            values.append(subset["proportion"].iloc[0])
        else:
            values.append(0)

    ax.bar(
        axes_order,
        values,
        bottom=bottom,
        label=category,
        color=colors[category]
    )

    bottom += np.array(values)


# ============================================================
# 6. Add 50% reference line
# ============================================================

ax.axhline(
    y=0.5,
    linestyle="--",
    linewidth=1,
    color="black",
    alpha=0.7
)

ax.text(
    x=len(axes_order)-3.6,
    y=0.48,
    s="50%",
    fontsize=9,
    va="bottom",
    ha="right"
)


# ============================================================
# 7. Formatting
# ============================================================

ax.set_ylim(0, 1)

ax.set_ylabel(
    "Percentage",
    fontsize=10
)

ax.set_xlabel("")

ax.set_title(
    "Comparison of Refined and Zero-shot Outputs",
    fontsize=11
)


# Wrapped x-axis labels
ax.set_xticks(range(len(axes_order)))
ax.set_xticklabels(
    axis_labels,
    fontsize=9
)


ax.set_yticks(np.linspace(0, 1, 6))

ax.set_yticklabels(
    [f"{int(x*100)}%" for x in np.linspace(0, 1, 6)],
    fontsize=9
)


ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=4,
    fontsize=9,
    frameon=False
)


plt.tight_layout()
plt.savefig("refinement_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Print metaphor IDs where zero-shot was rated higher
# ============================================================

# Get zero-shot > refined cases from the paired scores
zero_better = wide[
    wide["zero_shot"] > wide["refined"]
].copy()

# Add metaphor IDs back
metadata = (
    long_df[
        [
            "participant_id",
            "pair_num",
            "axis",
            "metaphor_id"
        ]
    ]
    .drop_duplicates()
)

zero_better = zero_better.merge(
    metadata,
    on=["participant_id", "pair_num", "axis"],
    how="left"
)

print(f"Number of cases where zero-shot scored higher: {len(zero_better)}")

print(
    zero_better[
        [
            "participant_id",
            "pair_num",
            "axis",
            "metaphor_id",
            "zero_shot",
            "refined"
        ]
    ]
    .sort_values(["metaphor_id", "axis"])
    .to_string(index=False)
)

In [ ]:
# ============================================================
# Find metaphors where zero-shot always beats refined
# ============================================================

# Get paired scores with metaphor IDs
comparison_df = wide.merge(
    long_df[
        [
            "participant_id",
            "pair_num",
            "axis",
            "metaphor_id"
        ]
    ].drop_duplicates(),
    on=["participant_id", "pair_num", "axis"],
    how="left"
)

# Determine winner for each individual rating
comparison_df["winner"] = np.where(
    comparison_df["zero_shot"] > comparison_df["refined"],
    "zero_shot",
    np.where(
        comparison_df["refined"] > comparison_df["zero_shot"],
        "refined",
        "tie"
    )
)

# Aggregate by metaphor
metaphor_summary = (
    comparison_df
    .groupby("metaphor_id")
    .agg(
        n_reviews=("winner", "count"),
        zero_better=("winner", lambda x: (x == "zero_shot").sum()),
        refined_better=("winner", lambda x: (x == "refined").sum()),
        ties=("winner", lambda x: (x == "tie").sum())
    )
    .reset_index()
)

# Keep metaphors where zero-shot wins every time
always_zero = metaphor_summary[
    (metaphor_summary["zero_better"] == metaphor_summary["n_reviews"])
]

print(
    f"Metaphors where zero-shot always beats refined: {len(always_zero)}"
)

print(always_zero.to_string(index=False))

In [ ]:
# ============================================================
# Find metaphors where refined always beats zero-shot
# ============================================================

# Keep metaphors where refined wins every individual rating
always_refined = metaphor_summary[
    (metaphor_summary["refined_better"] == metaphor_summary["n_reviews"])
]

print(
    f"Metaphors where refined always beats zero-shot: {len(always_refined)}"
)

print(always_refined.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. Prepare rating comparisons (AA + MC)
# ============================================================

comparison_df = wide.merge(
    long_df[
        [
            "participant_id",
            "pair_num",
            "axis",
            "metaphor_id"
        ]
    ].drop_duplicates(),
    on=["participant_id", "pair_num", "axis"],
    how="left"
)

comparison_df["winner"] = np.where(
    comparison_df["refined"] > comparison_df["zero_shot"],
    "refined",
    np.where(
        comparison_df["zero_shot"] > comparison_df["refined"],
        "zero_shot",
        "tie"
    )
)


# ============================================================
# 2. Function for unanimous winners
# ============================================================

def unanimous_results(df, axis_name):
    axis_df = df[df["axis"] == axis_name]

    summary = (
        axis_df
        .groupby("metaphor_id")
        .agg(
            n_reviews=("winner", "count"),
            refined_wins=("winner", lambda x: (x == "refined").sum()),
            zero_wins=("winner", lambda x: (x == "zero_shot").sum()),
            ties=("winner", lambda x: (x == "tie").sum())
        )
        .reset_index()
    )

    refined_always = summary[
        summary["refined_wins"] == summary["n_reviews"]
    ]

    zero_always = summary[
        summary["zero_wins"] == summary["n_reviews"]
    ]

    return refined_always, zero_always, summary


# ============================================================
# 3. AA and MC unanimous results
# ============================================================

for axis in ["AA", "MC"]:

    refined, zero, summary = unanimous_results(
        comparison_df,
        axis
    )

    print("\n==============================")
    print(axis)
    print("==============================")

    print(
        f"Refined unanimously better: {len(refined)} metaphors"
    )

    print(
        f"Zero-shot unanimously better: {len(zero)} metaphors"
    )


# ============================================================
# 4. Forced-choice unanimous results
# ============================================================

forced_df = (
    long_df[
        [
            "participant_id",
            "pair_num",
            "metaphor_id",
            "preferred_condition"
        ]
    ]
    .drop_duplicates()
)


forced_summary = (
    forced_df
    .groupby("metaphor_id")
    .agg(
        n_reviews=("preferred_condition", "count"),
        refined_wins=(
            "preferred_condition",
            lambda x: (x == "refined").sum()
        ),
        zero_wins=(
            "preferred_condition",
            lambda x: (x == "zero_shot").sum()
        )
    )
    .reset_index()
)


forced_refined_always = forced_summary[
    forced_summary["refined_wins"] == forced_summary["n_reviews"]
]

forced_zero_always = forced_summary[
    forced_summary["zero_wins"] == forced_summary["n_reviews"]
]


print("\n==============================")
print("Forced choice")
print("==============================")

print(
    f"Refined unanimously preferred: {len(forced_refined_always)} metaphors"
)

print(
    f"Zero-shot unanimously preferred: {len(forced_zero_always)} metaphors"
)

In [ ]:
# ============================================================
# Overall unanimous preference across AA + MC + forced choice
# ============================================================

# -----------------------------
# AA + MC unanimous results
# -----------------------------

rating_summary_all = (
    comparison_df
    .groupby(["metaphor_id", "axis"])
    .agg(
        n_reviews=("winner", "count"),
        refined_wins=("winner", lambda x: (x == "refined").sum()),
        zero_wins=("winner", lambda x: (x == "zero_shot").sum())
    )
    .reset_index()
)

# Pivot so AA and MC are separate columns
rating_pivot = rating_summary_all.pivot(
    index="metaphor_id",
    columns="axis",
    values=["n_reviews", "refined_wins", "zero_wins"]
)

rating_pivot.columns = [
    "_".join(col) for col in rating_pivot.columns
]

rating_pivot = rating_pivot.reset_index()


# -----------------------------
# Forced choice unanimous results
# -----------------------------

forced_summary = (
    forced_df
    .groupby("metaphor_id")
    .agg(
        forced_n_reviews=("preferred_condition", "count"),
        forced_refined_wins=(
            "preferred_condition",
            lambda x: (x == "refined").sum()
        ),
        forced_zero_wins=(
            "preferred_condition",
            lambda x: (x == "zero_shot").sum()
        )
    )
    .reset_index()
)


# -----------------------------
# Combine all three
# -----------------------------

overall = rating_pivot.merge(
    forced_summary,
    on="metaphor_id",
    how="inner"
)


# -----------------------------
# Check unanimous winners
# -----------------------------

overall["refined_unanimous_all"] = (
    (overall["refined_wins_AA"] == overall["n_reviews_AA"]) &
    (overall["refined_wins_MC"] == overall["n_reviews_MC"]) &
    (overall["forced_refined_wins"] == overall["forced_n_reviews"])
)


overall["zero_unanimous_all"] = (
    (overall["zero_wins_AA"] == overall["n_reviews_AA"]) &
    (overall["zero_wins_MC"] == overall["n_reviews_MC"]) &
    (overall["forced_zero_wins"] == overall["forced_n_reviews"])
)


# -----------------------------
# Results
# -----------------------------

n_refined_all = overall["refined_unanimous_all"].sum()
n_zero_all = overall["zero_unanimous_all"].sum()

print(
    f"Metaphors unanimously better for refined across AA + MC + forced choice: "
    f"{n_refined_all}"
)

print(
    f"Metaphors unanimously better for zero-shot across AA + MC + forced choice: "
    f"{n_zero_all}"
)

print(
    f"Total metaphors evaluated: {len(overall)}"
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter


# ============================================================
# Prepare data
# ============================================================

plot_df = long_df[
    long_df["axis"].isin(["AA", "MC"])
].copy()

axis_labels = {
    "AA": "Analogy Appropriateness",
    "MC": "Metaphor Consistency"
}


# ============================================================
# Plot overlaid density distributions
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4),
    sharey=True
)


scores = np.linspace(0, 10, 400)

colours = {
    "zero_shot": "#C44E52",
    "refined": "#4C72B0"
}

labels = {
    "zero_shot": "Zero-shot",
    "refined": "Refined"
}


for ax, axis in zip(axes, ["AA", "MC"]):

    subset = plot_df[
        plot_df["axis"] == axis
    ]

    densities = {}


    # ========================================================
    # Compute KDEs
    # ========================================================

    for condition in ["zero_shot", "refined"]:

        values = subset[
            subset["condition"] == condition
        ]["score"].dropna().values


        kde = gaussian_kde(values)

        density = kde(scores)

        # Scale area to 100%
        density = (
            density /
            np.trapezoid(density, scores)
            * 100
        )


        densities[condition] = (
            density,
            values.mean()
        )


    max_density = max(
        density.max()
        for density, _ in densities.values()
    )


    # ========================================================
    # Plot curves
    # ========================================================

    for condition in ["zero_shot", "refined"]:

        density, mean_score = densities[condition]


        ax.plot(
            scores,
            density,
            linewidth=2,
            color=colours[condition],
            label=labels[condition]
        )


        ax.fill_between(
            scores,
            density,
            alpha=0.2,
            color=colours[condition]
        )


        # Mean line
        ax.axvline(
            mean_score,
            linestyle="--",
            linewidth=1.5,
            color=colours[condition],
            alpha=0.8
        )


    # ========================================================
    # Mean annotations
    # ========================================================

    zero_mean = densities["zero_shot"][1]
    refined_mean = densities["refined"][1]


    ax.text(
        zero_mean - 0.15,
        max_density * 0.85,
        f"mean={zero_mean:.2f}",
        color=colours["zero_shot"],
        fontsize=8,
        ha="right",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.8,
            pad=1
        )
    )


    ax.text(
        refined_mean + 0.15,
        max_density * 0.65,
        f"mean={refined_mean:.2f}",
        color=colours["refined"],
        fontsize=8,
        ha="left",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.8,
            pad=1
        )
    )


    # ========================================================
    # Formatting
    # ========================================================

    ax.set_title(
        axis_labels[axis],
        fontsize=11
    )

    ax.set_xlabel(
        "Human score"
    )

    # Explicit 0–10 x-axis
    ax.set_xlim(
        0,
        10
    )

    ax.set_xticks(
        range(0, 11)
    )


    # Prevent clipping
    ax.set_ylim(
        0,
        max_density * 1.25
    )


    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )


axes[0].set_ylabel(
    "Percentage of ratings"
)

axes[0].yaxis.set_major_formatter(
    PercentFormatter(100)
)


# ============================================================
# Legend below entire figure
# ============================================================

legend_handles = [
    Line2D(
        [0],
        [0],
        color=colours["zero_shot"],
        linewidth=2,
        label="Zero-shot"
    ),

    Line2D(
        [0],
        [0],
        color=colours["refined"],
        linewidth=2,
        label="Refined"
    ),

    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        linewidth=1.5,
        label="Mean score"
    )
]


fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=3,
    frameon=False
)


# ============================================================
# Final layout
# ============================================================

plt.suptitle(
    "Distribution of Human Evaluation Scores",
    fontsize=12
)


plt.tight_layout(
    rect=[0, 0.08, 1, 1]
)

plt.tight_layout()
plt.savefig("score_distributions.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# Keep only valid group IDs (1-80) and one response per group
# ============================================================

df_raw["group_id"] = pd.to_numeric(
    df_raw["group_id"],
    errors="coerce"
)

unique_group_df = (
    df_raw[
        df_raw["group_id"].between(1, 80)
    ]
    .drop_duplicates(
        subset=["group_id"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Original responses:", len(df_raw))
print("After filtering valid group IDs and duplicates:", len(unique_group_df))
print("Unique group IDs:", unique_group_df["group_id"].nunique())

print("\nMissing group IDs:")
missing_ids = sorted(
    set(range(1, 81)) -
    set(unique_group_df["group_id"].astype(int))
)
print(missing_ids)

# ============================================================
# Age overview (Q43)
# ============================================================

ages = pd.to_numeric(
    unique_group_df["Q43"],
    errors="coerce"
).dropna()

print("========== AGE (Q43) ==========")
print(f"Number of valid responses: {len(ages)}")
print(f"Minimum age: {ages.min()}")
print(f"Maximum age: {ages.max()}")
print(f"Mean age: {ages.mean():.2f}")
print(f"Median age: {ages.median()}")

print("\nAge frequencies:")
print(
    ages.value_counts()
    .sort_index()
)


# ============================================================
# Gender overview (Q44)
# ============================================================

gender = (
    unique_group_df["Q44"]
    .dropna()
    .value_counts()
)

print("\n\n========== GENDER (Q44) ==========")
print(gender)

print("\nGender percentages:")
print(
    (gender / gender.sum() * 100)
    .round(1)
)


# ============================================================
# Education overview (Q45)
# ============================================================

education = (
    unique_group_df["Q45"]
    .dropna()
    .value_counts()
)

print("\n\n========== EDUCATION (Q45) ==========")
print(education)

print("\nEducation percentages:")
print(
    (education / education.sum() * 100)
    .round(1)
)

In [ ]:
from scipy.stats import pearsonr, spearmanr


# ============================================================
# Load VLM scores
# ============================================================

vlm_zero = pd.read_csv("../evaluation/results/stricter_judge_zero.csv")
vlm_refined = pd.read_csv("../evaluation/results/stricter_judge_last.csv")

vlm_zero["condition"] = "zero_shot"
vlm_refined["condition"] = "refined"

vlm = pd.concat(
    [vlm_zero, vlm_refined],
    ignore_index=True
)

# Keep ids as strings (003, 015, ...)
vlm["id"] = vlm["id"].astype(str).str.zfill(3)


# ============================================================
# Aggregate human scores
# ============================================================

human = (
    long_df
    .groupby(
        ["metaphor_id", "condition", "axis"],
        as_index=False
    )["score"]
    .mean()
)

human["metaphor_id"] = (
    human["metaphor_id"]
    .astype(str)
    .str.zfill(3)
)


# ============================================================
# Map axis names
# ============================================================

judge_columns = {
    "MC": "judge_metaphor_consistency",
    "AA": "judge_analogy_appropriateness"
}


# ============================================================
# Compute correlations
# ============================================================

results = []

for condition in ["zero_shot", "refined"]:

    vlm_subset = vlm[
        vlm["condition"] == condition
    ]

    for axis in ["MC", "AA"]:

        human_subset = human[
            (human["condition"] == condition) &
            (human["axis"] == axis)
        ]

        merged = human_subset.merge(
            vlm_subset,
            left_on="metaphor_id",
            right_on="id"
        )

        human_scores = merged["score"]
        vlm_scores = merged[
            judge_columns[axis]
        ]

        pearson_r, pearson_p = pearsonr(
            human_scores,
            vlm_scores
        )

        spearman_rho, spearman_p = spearmanr(
            human_scores,
            vlm_scores
        )

        results.append({
            "condition": condition,
            "axis": axis,
            "n": len(merged),
            "Pearson r": pearson_r,
            "Pearson p": pearson_p,
            "Spearman rho": spearman_rho,
            "Spearman p": spearman_p
        })


results_df = pd.DataFrame(results)

print(results_df.round(3))

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# ============================================================
# 1. Load VLM judge scores
# ============================================================

zero = pd.read_csv("../evaluation/results/stricter_judge_zero.csv")
refined = pd.read_csv("../evaluation/results/stricter_judge_last.csv")

# Ensure IDs match the human data
zero["id"] = zero["id"].astype(str).str.zfill(3)
refined["id"] = refined["id"].astype(str).str.zfill(3)

# ============================================================
# 2. Compute overall VLM score
# ============================================================

judge_axes = [
    "judge_metaphor_consistency",
    "judge_analogy_appropriateness",
    "judge_conceptual_integration"
]

zero["overall_score"] = zero[judge_axes].mean(axis=1)
refined["overall_score"] = refined[judge_axes].mean(axis=1)

# ============================================================
# 3. Determine preferred image for each metaphor
# ============================================================

vlm_pref = (
    zero[["id", "overall_score"]]
    .rename(columns={"overall_score": "zero_score"})
    .merge(
        refined[["id", "overall_score"]]
        .rename(columns={"overall_score": "refined_score"}),
        on="id"
    )
)

# Remove exact ties (rare)
vlm_pref = vlm_pref[
    vlm_pref["zero_score"] != vlm_pref["refined_score"]
].copy()

vlm_pref["vlm_choice"] = (
    vlm_pref["refined_score"] > vlm_pref["zero_score"]
).map({
    True: "refined",
    False: "zero_shot"
})

print(f"Metaphors with VLM preference: {len(vlm_pref)}")

# ============================================================
# 4. Recover every human forced-choice judgement
# ============================================================

records = []

for pair_idx in range(N_PAIRS):

    pair_num = pair_idx + 1

    choice_col = choice_cols[pair_idx]
    id_col = pair_id_cols[pair_idx]

    refined_letter = REFINED_LETTER_BY_PAIR[pair_idx]

    for _, row in df.iterrows():

        if pd.isna(row[choice_col]):
            continue

        if row[choice_col] == f"Image {refined_letter}":
            human_choice = "refined"
        else:
            human_choice = "zero_shot"

        records.append({
            "participant_id": row["participant_id"],
            "metaphor_id": str(row[id_col]).zfill(3),
            "human_choice": human_choice
        })

human_pref = pd.DataFrame(records)

print(f"Human judgements: {len(human_pref)}")

# ============================================================
# 5. Merge VLM + human
# ============================================================

comparison = human_pref.merge(
    vlm_pref[["id", "vlm_choice"]],
    left_on="metaphor_id",
    right_on="id",
    how="inner"
)

print(f"Matched judgements: {len(comparison)}")

# ============================================================
# 6. Cohen's kappa
# ============================================================

kappa = cohen_kappa_score(
    comparison["human_choice"],
    comparison["vlm_choice"]
)

agreement = (
    comparison["human_choice"] ==
    comparison["vlm_choice"]
).mean()

print("\n==============================")
print("Agreement")
print("==============================")
print(f"Cohen's κ = {kappa:.3f}")
print(f"Raw agreement = {agreement:.1%}")

# ============================================================
# 7. Confusion matrix
# ============================================================

cm = pd.DataFrame(
    confusion_matrix(
        comparison["human_choice"],
        comparison["vlm_choice"],
        labels=["refined", "zero_shot"]
    ),
    index=[
        "Human: Refined",
        "Human: Zero-shot"
    ],
    columns=[
        "VLM: Refined",
        "VLM: Zero-shot"
    ]
)

print("\nConfusion matrix")
print(cm)

In [ ]:
import krippendorff

forced_df = forced_df.dropna(subset=["metaphor_id"]).copy()

forced_df["metaphor_id"] = (
    forced_df["metaphor_id"]
    .astype(str)
    .str.zfill(3)
)

forced_df["rater"] = (
    forced_df.groupby("metaphor_id")
    .cumcount()
)

matrix = forced_df.pivot(
    index="metaphor_id",
    columns="rater",
    values="choice_num"
)

alpha = krippendorff.alpha(
    reliability_data=matrix.to_numpy().T,
    level_of_measurement="nominal"
)

print(f"Krippendorff's α = {alpha:.3f}")

In [ ]:
import krippendorff

for axis in ["AA", "MC"]:

    ratings = (
        long_df[long_df["axis"] == axis]
        .dropna(subset=["score", "metaphor_id"])
        .copy()
    )

    ratings["rater"] = ratings.groupby("metaphor_id").cumcount()

    matrix = ratings.pivot(
        index="metaphor_id",
        columns="rater",
        values="score"
    )

    alpha = krippendorff.alpha(
        reliability_data=matrix.to_numpy().T,
        level_of_measurement="ordinal"
    )

    print(f"{axis}: Krippendorff's α = {alpha:.3f}")

In [ ]:
long_df.groupby(["metaphor_id", "axis"]).size().value_counts()

In [ ]:
long_df[
    long_df["axis"]=="AA"
].groupby("metaphor_id")["score"].agg(["count", "std"])

In [ ]:
import pandas as pd
import krippendorff


# ============================================================
# 1. Create refined vs zero-shot differences
# ============================================================

comparison_df = (
    long_df
    .pivot_table(
        index=["participant_id", "metaphor_id", "axis"],
        columns="condition",
        values="score"
    )
    .reset_index()
)

comparison_df["difference"] = (
    comparison_df["refined"]
    -
    comparison_df["zero_shot"]
)


# ============================================================
# 2. Convert to categorical preference
# ============================================================

def classify_improvement(row):

    if row["difference"] > 0:
        return "refined_better"

    elif row["difference"] < 0:
        return "zero_better"

    else:
        return "equal"


comparison_df["preference"] = (
    comparison_df
    .apply(classify_improvement, axis=1)
)


# ============================================================
# 3. Calculate Krippendorff alpha per axis
# ============================================================

for axis in ["AA", "MC"]:

    subset = comparison_df[
        comparison_df["axis"] == axis
    ].copy()

    # encode categories
    mapping = {
        "zero_better": 0,
        "equal": 1,
        "refined_better": 2
    }

    subset["preference_num"] = (
        subset["preference"]
        .map(mapping)
    )


    # assign rater number within each metaphor
    subset["rater"] = (
        subset
        .groupby("metaphor_id")
        .cumcount()
    )


    matrix = subset.pivot(
        index="metaphor_id",
        columns="rater",
        values="preference_num"
    )


    alpha = krippendorff.alpha(
        reliability_data=matrix.to_numpy().T,
        level_of_measurement="nominal"
    )


    print("==============================")
    print(axis)
    print("==============================")
    print(f"Krippendorff's α = {alpha:.3f}")


    # descriptive statistics
    print("\nPreference distribution:")
    print(
        subset["preference"]
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
    )

In [ ]:
for axis in ["AA", "MC"]:

    subset = comparison_df[
        comparison_df["axis"] == axis
    ]

    majority = (
        subset
        .groupby("metaphor_id")["preference"]
        .agg(
            lambda x:
            x.value_counts().iloc[0]
            /
            len(x)
        )
    )

    print("\n", axis)

    print(
        f"Mean majority agreement: "
        f"{majority.mean():.1%}"
    )

    print(
        f"Unanimous metaphors: "
        f"{(majority == 1).sum()}/"
        f"{len(majority)}"
    )